# Notebook 05: Novel Candidate Screening

**Project:** GNN-Based Battery Voltage Predictor  

---

## Overview

Using the best trained model (M3GNet), this notebook screens novel Li-containing
crystal structures that do not yet have computed voltages in the Materials Project.

Screening pipeline:

1. Query Materials Project for Li-containing structures with:
   - e_above_hull less than 0.05 eV/atom (thermodynamically stable)
   - No existing electrode entry (not already in our training data)
   - Contains Li (working ion)

2. Run voltage inference with the fine-tuned M3GNet on all candidates

3. Filter and rank by:
   - Predicted voltage greater than 2.5 V vs Li/Li+ (practical cathode range)
   - Estimated stability (e_above_hull less than 50 meV/atom)
   - Formula does not contain rare/toxic elements (Hg, Tl, Pb, Cd) for feasibility

4. Output top 50 candidates as `results/top_candidates.csv`

5. Validate top 5 against any available DFT data or literature

**Note:** Inference voltage predictions for unstudied structures are extrapolations;
DFT validation is required before experimental synthesis decisions.

In [ ]:
import sys
import json
import pickle
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import CrystalTransformer
from src.utils import structure_to_graph, get_chemistry_family, set_seed
from src.evaluate import PALETTE
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.data import Data

set_seed(42)

API_KEY = 'py7ZFSvHtzOMRc0pMB3GHBhM5m1pohBF'

DATA_DIR    = project_root / 'data'
MODELS_DIR  = project_root / 'models'
RESULTS_DIR = project_root / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')


## 1. Query Materials Project for Novel Candidates

We query stable Li-containing compounds that are not in our training dataset.
The filter `e_above_hull < 0.05 eV/atom` selects entries on or very close to
the convex hull of thermodynamic stability. We also request the crystal structure
so we can featurize directly for GNN inference.

In [ ]:
with open(DATA_DIR / 'splits.pkl', 'rb') as f:
    splits = pickle.load(f)

known_battery_ids = set()
known_formulas = set()
for split_name in ['train', 'val', 'test']:
    for e in splits[split_name]:
        known_formulas.add(e['formula'])
        known_battery_ids.add(e['battery_id'])

print(f'Known training formulas: {len(known_formulas)}')


In [ ]:
from mp_api.client import MPRester
from pymatgen.core import Structure, Composition

CANDIDATE_CACHE = DATA_DIR / 'screening_candidates.pkl'
EXCLUDE_ELEMENTS = {'Hg', 'Tl', 'Pb', 'Cd', 'Cs', 'Rb', 'Fr', 'Ra', 'At', 'Po'}

if CANDIDATE_CACHE.exists():
    print(f'Loading cached candidates from {CANDIDATE_CACHE}')
    with open(CANDIDATE_CACHE, 'rb') as f:
        candidates = pickle.load(f)
else:
    print('Querying Materials Project for novel Li-containing structures...')
    candidates = []

    with MPRester(api_key=API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            elements=['Li'],
            energy_above_hull=(0, 0.05),
            fields=[
                'material_id', 'formula_pretty', 'energy_above_hull',
                'structure', 'nsites', 'formation_energy_per_atom',
            ],
        )

    print(f'Retrieved {len(docs)} raw candidates')

    for doc in docs:
        formula = getattr(doc, 'formula_pretty', '') or ''
        if formula in known_formulas:
            continue
        try:
            comp = Composition(formula)
            if comp.get('Li', 0) == 0:
                continue
            if any(str(el) in EXCLUDE_ELEMENTS for el in comp.elements):
                continue
        except Exception:
            continue

        structure = getattr(doc, 'structure', None)
        if structure is None:
            continue

        candidates.append({
            'material_id': str(getattr(doc, 'material_id', '')),
            'formula': formula,
            'chemistry_family': get_chemistry_family(formula),
            'energy_above_hull': float(getattr(doc, 'energy_above_hull', 0) or 0),
            'formation_energy_per_atom': float(getattr(doc, 'formation_energy_per_atom', 0) or 0),
            'nsites': int(getattr(doc, 'nsites', 0) or 0),
            'structure_dict': structure.as_dict(),
        })

    print(f'Novel stable Li candidates: {len(candidates)}')

    with open(CANDIDATE_CACHE, 'wb') as f:
        pickle.dump(candidates, f)
    print(f'Cached to {CANDIDATE_CACHE}')


In [ ]:
df_cand = pd.DataFrame([{
    'material_id': c['material_id'],
    'formula': c['formula'],
    'energy_above_hull_meV': c['energy_above_hull'] * 1000,
    'chemistry_family': c['chemistry_family'],
    'nsites': c['nsites'],
} for c in candidates])

print(f'Candidate dataset: {len(df_cand)} entries')
print(df_cand['chemistry_family'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(df_cand['energy_above_hull_meV'], bins=40, color='#4CAF50', alpha=0.8, edgecolor='none')
axes[0].set_xlabel('E above hull (meV/atom)')
axes[0].set_ylabel('Count')
axes[0].set_title('Stability of Screening Candidates')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

chem_counts = df_cand['chemistry_family'].value_counts()
chem_colors = [PALETTE.get(f, '#607D8B') for f in chem_counts.index]
axes[1].bar(range(len(chem_counts)), chem_counts.values, color=chem_colors, alpha=0.85, edgecolor='none')
axes[1].set_xticks(range(len(chem_counts)))
axes[1].set_xticklabels(chem_counts.index, rotation=25, ha='right')
axes[1].set_ylabel('Count')
axes[1].set_title('Candidates by Chemistry Family')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig05_candidate_overview.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: fig05_candidate_overview.png')


## 2. Voltage Inference with Best Model (CrystalTransformer)

We run the CrystalTransformer on each candidate structure to predict the Li insertion voltage.


In [ ]:
# Load CrystalTransformer (best model)
ct_model = CrystalTransformer(node_dim=9, edge_dim=64, hidden_dim=128,
                               n_conv=4, heads=4, dropout=0.1)
ct_model.load_state_dict(torch.load(MODELS_DIR / 'transformer_best.pt', map_location=device))
ct_model = ct_model.to(device)
ct_model.eval()
print(f'CrystalTransformer loaded: {ct_model.count_parameters():,} parameters')


In [ ]:
# Convert candidate structures to PyG graphs
print(f'Converting {len(candidates)} structures to graphs (cutoff=5.0 A)...')
candidate_graphs = []
valid_indices = []

for i, cand in enumerate(candidates):
    try:
        structure = Structure.from_dict(cand['structure_dict'])
        graph = structure_to_graph(structure, cutoff=5.0, n_gbf_bins=64)
        candidate_graphs.append(graph)
        valid_indices.append(i)
    except Exception:
        continue
    if (i + 1) % 500 == 0:
        print(f'  Processed {i+1}/{len(candidates)}...')

print(f'Built {len(candidate_graphs)} valid graphs from {len(candidates)} candidates')


In [ ]:
# Batch inference
loader = PyGLoader(candidate_graphs, batch_size=64, shuffle=False, num_workers=0)
all_preds = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)
        preds = ct_model(batch)
        all_preds.append(preds.cpu().numpy())

predicted_voltages = np.concatenate(all_preds)
print(f'Inference complete: {len(predicted_voltages)} predictions')
print(f'Predicted voltage range: {predicted_voltages.min():.2f} to {predicted_voltages.max():.2f} V')


In [ ]:
# Build results dataframe
screening_rows = []
for graph_idx, (orig_idx, pred_v) in enumerate(zip(valid_indices, predicted_voltages)):
    cand = candidates[orig_idx]
    try:
        comp = Composition(cand['formula'])
        mw = comp.weight
        n_li = comp.get_el_amt_dict().get('Li', 0)
        est_capacity = n_li * 26802 / mw if mw > 0 else 0
    except Exception:
        est_capacity = 0

    screening_rows.append({
        'material_id': cand['material_id'],
        'formula': cand['formula'],
        'chemistry_family': cand['chemistry_family'],
        'predicted_voltage_V': float(pred_v),
        'est_capacity_mAh_g': est_capacity,
        'est_energy_density_Wh_kg': float(pred_v) * est_capacity,
        'energy_above_hull_meV': cand['energy_above_hull'] * 1000,
        'formation_energy_eV_atom': cand['formation_energy_per_atom'],
        'nsites': cand['nsites'],
    })

df_screening = pd.DataFrame(screening_rows)

# Filter: voltage 2-6 V (physically realistic), stability < 50 meV/atom
df_filtered = df_screening[
    (df_screening['predicted_voltage_V'] >= 2.0) &
    (df_screening['predicted_voltage_V'] <= 6.0) &
    (df_screening['energy_above_hull_meV'] < 50)
].copy()
df_filtered = df_filtered.sort_values('predicted_voltage_V', ascending=False).reset_index(drop=True)

print(f'All candidates screened: {len(df_screening)}')
print(f'Passing voltage (2-6 V) + stability (< 50 meV) filters: {len(df_filtered)}')
print()
print(df_filtered.head(10)[['formula', 'chemistry_family', 'predicted_voltage_V',
                              'est_capacity_mAh_g', 'energy_above_hull_meV']].to_string(index=False))


In [ ]:
top50 = df_filtered.head(50).copy()
top50['rank'] = range(1, len(top50) + 1)

for col in ['predicted_voltage_V', 'est_capacity_mAh_g', 'est_energy_density_Wh_kg', 'energy_above_hull_meV']:
    top50[col] = top50[col].round(3)

top50.to_csv(RESULTS_DIR / 'top_candidates.csv', index=False)
print(f'Saved top 50 candidates to top_candidates.csv')
print()
print(top50[['rank', 'formula', 'chemistry_family', 'predicted_voltage_V',
             'est_capacity_mAh_g', 'energy_above_hull_meV']].head(20).to_string(index=False))


## 3. Screening Results Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for fam, grp in df_screening.groupby('chemistry_family'):
    # Only show physically-plausible voltage range
    grp_filt = grp[(grp['predicted_voltage_V'] >= 0) & (grp['predicted_voltage_V'] <= 7)]
    ax.scatter(grp_filt['est_capacity_mAh_g'], grp_filt['predicted_voltage_V'],
               s=10, alpha=0.4, label=fam,
               color=PALETTE.get(fam, '#607D8B'), linewidths=0)

# Highlight top 10
for _, row in top50.head(10).iterrows():
    ax.scatter(row['est_capacity_mAh_g'], row['predicted_voltage_V'],
               s=80, color='gold', marker='*', zorder=5,
               edgecolors='black', linewidths=0.5)
    ax.annotate(row['formula'], (row['est_capacity_mAh_g'], row['predicted_voltage_V']),
                fontsize=7, ha='left', va='bottom')

ax.axhline(2.0, color='grey', linestyle=':', lw=1.0, alpha=0.7, label='2.0 V cutoff')
ax.set_xlabel('Estimated Gravimetric Capacity (mAh/g)')
ax.set_ylabel('Predicted Voltage (V vs Li/Li+)')
ax.set_title('Novel Candidate Screening: Predicted Voltage vs Estimated Capacity')
ax.legend(title='Chemistry', fontsize=8, frameon=True, framealpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig05_screening_scatter.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: fig05_screening_scatter.png')

# Voltage distribution by chemistry
fig, ax = plt.subplots(figsize=(7, 4))
df_valid = df_screening[(df_screening['predicted_voltage_V'] >= 0) & 
                         (df_screening['predicted_voltage_V'] <= 7)]
for fam in df_valid['chemistry_family'].unique():
    grp = df_valid[df_valid['chemistry_family'] == fam]['predicted_voltage_V']
    ax.hist(grp, bins=30, alpha=0.5, label=fam, color=PALETTE.get(fam, '#607D8B'),
            edgecolor='none', density=True)
ax.set_xlabel('Predicted Voltage (V vs Li/Li+)')
ax.set_ylabel('Density')
ax.set_title('Predicted Voltage Distribution by Chemistry Family')
ax.legend(title='Chemistry', fontsize=8, frameon=True, framealpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig05_voltage_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: fig05_voltage_distribution.png')


## 4. Validation of Top 5 Candidates

For the top 5 predicted candidates, we check whether any existing DFT data or
literature voltages are available in the Materials Project under a different entry
(e.g., as an electrode material with a slightly different composition).

This acts as a sanity check on the model's extrapolative performance.

In [ ]:
top5 = top50.head(5)
print('Top 5 predicted candidates:')
print(top5[['rank', 'formula', 'chemistry_family', 'predicted_voltage_V',
             'est_capacity_mAh_g', 'energy_above_hull_meV']].to_string(index=False))

# Cross-reference with known electrode database for any matches
with open(DATA_DIR / 'battery_dataset.json') as f:
    all_battery = json.load(f)

known_voltages = {e['formula']: e['average_voltage'] for e in all_battery}
validation_results = []
for _, row in top5.iterrows():
    dft_v = known_voltages.get(row['formula'])
    validation_results.append({
        'formula': row['formula'],
        'predicted_voltage_V': row['predicted_voltage_V'],
        'dft_voltage_V': dft_v if dft_v else 'N/A (novel)',
        'abs_error_V': round(abs(dft_v - row['predicted_voltage_V']), 3) if dft_v else 'N/A',
    })

val_df = pd.DataFrame(validation_results)
print()
print('Validation against known DFT data (if available):')
print(val_df.to_string(index=False))
val_df.to_csv(RESULTS_DIR / 'top5_validation.csv', index=False)
print('Saved: top5_validation.csv')


In [ ]:
# Save full screening results
df_screening.to_csv(RESULTS_DIR / 'screening_all.csv', index=False)

print('Screening complete.')
print(f'  Total candidates processed: {len(df_screening)}')
print(f'  Passing voltage + stability filters: {len(df_filtered)}')
print(f'  Top 50 saved to: results/top_candidates.csv')
print(f'  Full results: results/screening_all.csv')
print(f'')
print('Proceed to 06_dashboard.ipynb')
